# 01 — Instance Parser and ILP (Seyon)

**Day 1 goal:** all 24 benchmark instances load correctly and are checked against `data/best_known.csv`.

Dataset: Nurse Rostering Benchmark Instances 1-24 (Curtois & Qu, University of Nottingham),
<https://www.schedulingbenchmarks.org/nrp/>. Format guide:
<https://www.schedulingbenchmarks.org/nrp/instances1_24.html>. The parser lives in
`rostering_common.py` at the repository root, so it can be imported unchanged from every notebook
(`01_ilp.ipynb`, `02_ga.ipynb`, `03_comparison.ipynb`).

## File format, section by section

Each `data/raw/InstanceN.txt` is a plain-text file. Lines starting with `#` are comments and blank
lines are ignored; every other line belongs to the section named by the most recent `SECTION_...`
header. Below is what each section means and how it is stored in the `Instance` dataclass.

| Section | Fields (comma-separated) | Meaning | Stored as |
|---|---|---|---|
| `SECTION_HORIZON` | `n_days` | Length of the planning period in days. Day 0 is always a Monday. | `int` — `Instance.n_days` |
| `SECTION_SHIFTS` | `shift_id, length_minutes, forbidden_next` | One shift type: its id (can be several characters, e.g. `d1`, `a2`), its length in minutes, and the `\|`-separated list of shift ids that must **not** be worked the day after this one (the list can be empty, e.g. `D,480,` for a shift with no restriction). | `shift_ids: list[str]` (keeps file order), `shift_len: dict[str,int]`, `forbidden_next: dict[str, set[str]]` |
| `SECTION_STAFF` | `nurse_id, max_shifts, max_minutes, min_minutes, max_consec, min_consec, min_days_off, max_weekends` | One nurse's contract. `max_shifts` is itself a `\|`-separated list `shift=count` (e.g. `E=14\|D=14\|L=0`); a count of 0 means the nurse can never work that shift type (used to encode skills). The rest are single integers: total-minutes window, longest and shortest allowed working block, shortest allowed rest block, and the cap on weekends worked. | `nurse_ids: list[str]`, `max_shifts: dict[str, dict[str,int]]`, and one `dict[str,int]` per remaining field |
| `SECTION_DAYS_OFF` | `nurse_id, day, day, ...` | Days on which this nurse must not be assigned any shift (hard rule). A nurse can have zero listed days, in which case the line for them is simply absent. | `days_off: dict[str, set[int]]` (every nurse gets an entry, even an empty set) |
| `SECTION_SHIFT_ON_REQUESTS` | `nurse_id, day, shift_id, weight` | The nurse asks to work this shift on this day; not granting it costs `weight` in the objective. | `on_requests: list[tuple[str,int,str,int]]` |
| `SECTION_SHIFT_OFF_REQUESTS` | `nurse_id, day, shift_id, weight` | The nurse asks **not** to work this shift on this day; assigning it anyway costs `weight`. | `off_requests: list[tuple[str,int,str,int]]` |
| `SECTION_COVER` | `day, shift_id, required, weight_under, weight_over` | How many nurses are required on this day for this shift, and the penalty per nurse for being short (`weight_under`) or over (`weight_over`). | `cover: dict[tuple[int,str], tuple[int,int,int]]` |

Traps to watch for (all handled by `load_instance`, see `rostering_common.py`): a shift id can be
more than one character; a forbidden-next list can be empty; `max_shifts` entries can appear in any
order and are not necessarily one-per-shift-type; comment and blank lines can appear between any two
data lines, not just at section boundaries.

In [1]:
import pandas as pd

from rostering_common import load_instance

inst1 = load_instance("data/raw/Instance1.txt")
print(f"name           = {inst1.name}")
print(f"n_days         = {inst1.n_days}  (n_weeks = {inst1.n_weeks})")
print(f"shift_ids      = {inst1.shift_ids}")
print(f"n_nurses       = {inst1.n_nurses}")
print(f"nurse_ids      = {inst1.nurse_ids}")
print(f"max_shifts['A']= {inst1.max_shifts['A']}")
print(f"days_off['A']  = {inst1.days_off['A']}")
print(f"cover[(0,'D')] = {inst1.cover[(0, 'D')]}   # (required, weight_under, weight_over)")
print(f"# on-requests  = {len(inst1.on_requests)}, # off-requests = {len(inst1.off_requests)}")


name           = Instance1
n_days         = 14  (n_weeks = 2)
shift_ids      = ['D']
n_nurses       = 8
nurse_ids      = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
max_shifts['A']= {'D': 14}
days_off['A']  = {0}
cover[(0,'D')] = (5, 100, 1)   # (required, weight_under, weight_over)
# on-requests  = 21, # off-requests = 5


## Test: all 24 instances load, and their sizes match `data/best_known.csv`

For every instance, `n_nurses`, `n_weeks` and `n_shift_types` (from the parsed file) must equal the
`employees`, `weeks` and `shift_types` columns of `data/best_known.csv` (the numbers the benchmark
page itself publishes for each instance).

In [2]:
best_known = pd.read_csv("data/best_known.csv").set_index("instance")

rows = []
mismatches = []
for i in range(1, 25):
    name = f"Instance{i}"
    inst = load_instance(f"data/raw/{name}.txt")
    expected = best_known.loc[name]
    ok_nurses = inst.n_nurses == expected["employees"]
    ok_weeks = inst.n_weeks == expected["weeks"]
    ok_shifts = inst.n_shift_types == expected["shift_types"]
    rows.append({
        "instance": name,
        "n_days": inst.n_days,
        "n_weeks": inst.n_weeks,
        "n_nurses": inst.n_nurses,
        "n_shift_types": inst.n_shift_types,
        "n_binaries": inst.n_nurses * inst.n_days * inst.n_shift_types,
        "sizes_match_best_known": ok_nurses and ok_weeks and ok_shifts,
    })
    if not (ok_nurses and ok_weeks and ok_shifts):
        mismatches.append(name)

sizes = pd.DataFrame(rows).set_index("instance")
display(sizes)

print()
assert len(sizes) == 24, f"expected 24 instances, loaded {len(sizes)}"
assert not mismatches, f"size mismatch against best_known.csv for: {mismatches}"
assert sizes["sizes_match_best_known"].all()
print("PASS: 24 of 24 files load, and n_nurses / n_weeks / n_shift_types match data/best_known.csv for every one.")


,n_days,n_weeks,n_nurses,n_shift_types,n_binaries,sizes_match_best_known
instance,,,,,,
Instance1,14,2,8,1,112,True
Instance2,14,2,14,2,392,True
Instance3,14,2,20,3,840,True
Instance4,28,4,10,2,560,True
Instance5,28,4,16,2,896,True
Instance6,28,4,18,3,1512,True
Instance7,28,4,20,3,1680,True
Instance8,28,4,30,4,3360,True
Instance9,28,4,36,4,4032,True



PASS: 24 of 24 files load, and n_nurses / n_weeks / n_shift_types match data/best_known.csv for every one.


## Try to break it — five edge cases

Five small, hand-written instance snippets, each targeting one thing the real files do that a naive
parser could get wrong. Each is written to a temporary file and parsed with the same
`load_instance` used above — no special-casing.

In [3]:
import tempfile
import os

def parse_snippet(text: str):
    """Write text to a temp file and parse it, like a real InstanceN.txt."""
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as f:
        f.write(text)
        path = f.name
    try:
        return load_instance(path)
    finally:
        os.remove(path)


edge_case_1 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
# a shift with NO forbidden-next entry at all (trailing comma, nothing after it)
D,480,

SECTION_STAFF
A,D=7,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,D,1,100,1
"""
inst = parse_snippet(edge_case_1)
assert inst.forbidden_next["D"] == set(), "empty forbidden-next list should parse to an empty set"
assert inst.days_off["A"] == set(), "a nurse with no SECTION_DAYS_OFF line should get an empty set, not a KeyError"
print("1. Empty forbidden-next list + nurse absent from SECTION_DAYS_OFF -> OK")


edge_case_2 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
# multi-letter shift ids, as the format guide allows (e.g. d1, a2)
d1,480,a2
a2,600,d1

SECTION_STAFF
A,d1=5|a2=2,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,d1,1,100,1
"""
inst = parse_snippet(edge_case_2)
assert inst.shift_ids == ["d1", "a2"]
assert inst.forbidden_next["d1"] == {"a2"}
assert inst.max_shifts["A"] == {"d1": 5, "a2": 2}
print("2. Multi-letter shift ids (d1, a2) -> OK")


edge_case_3 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
E,480,
D,480,E
L,480,E|D

SECTION_STAFF
# a shift count of 0 (nurse can never work that shift type)
A,E=7|D=0|L=0,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,E,1,100,1
"""
inst = parse_snippet(edge_case_3)
assert inst.max_shifts["A"]["D"] == 0 and inst.max_shifts["A"]["L"] == 0
print("3. MaxShifts entry of 0 for several shift types (E=7|D=0|L=0) -> OK")


edge_case_4 = """\
SECTION_HORIZON
# a comment line dropped in between two data lines of the same section
7

SECTION_SHIFTS
D,480,

SECTION_STAFF
A,D=7,3360,3360,7,1,1,1
# a comment between two staff lines
B,D=7,3360,3360,7,1,1,1

SECTION_DAYS_OFF
A,2,5
# comment
B,0

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,D,2,100,1
"""
inst = parse_snippet(edge_case_4)
assert inst.nurse_ids == ["A", "B"]
assert inst.days_off["A"] == {2, 5} and inst.days_off["B"] == {0}
print("4. Comment lines interleaved between data lines within a section -> OK")


edge_case_5 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
E,480,
D,480,E
L,480,E|D

SECTION_STAFF
A,E=7|D=7|L=7,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS
A,1,E,3
A,3,L,1

SECTION_SHIFT_OFF_REQUESTS
A,5,D,2

SECTION_COVER
0,E,1,100,1
0,D,1,100,1
0,L,1,100,1
"""
inst = parse_snippet(edge_case_5)
assert inst.on_requests == [("A", 1, "E", 3), ("A", 3, "L", 1)]
assert inst.off_requests == [("A", 5, "D", 2)]
assert len(inst.cover) == 3
print("5. Several shift types with both on- and off-requests, and one cover row per shift -> OK")

print()
print("All 5 edge cases handled correctly by the same load_instance() used on the real 24 files.")


1. Empty forbidden-next list + nurse absent from SECTION_DAYS_OFF -> OK
2. Multi-letter shift ids (d1, a2) -> OK
3. MaxShifts entry of 0 for several shift types (E=7|D=0|L=0) -> OK
4. Comment lines interleaved between data lines within a section -> OK
5. Several shift types with both on- and off-requests, and one cover row per shift -> OK

All 5 edge cases handled correctly by the same load_instance() used on the real 24 files.


## Day 1 status

- [x] `rostering_common.py` created with `load_instance(path) -> Instance`.
- [x] All 24 of 24 files load; `n_nurses` / `n_weeks` / `n_shift_types` match `data/best_known.csv` for every one.
- [x] Five edge cases (empty forbidden-next, multi-letter shift ids, a zero `MaxShifts` entry,
      interleaved comments, multiple requests/cover rows) parse correctly.

**Finished when:** 24 of 24 files load and the size test passes — done above.

Next (Day 2): shared roster format (`numpy` int array, `n_nurses x n_days`, 0 = day off,
`k` = the `k`-th shift in `shift_ids` order), then the first ILP model (C1-C4, C9, C10) in this
notebook, alongside the independent checker (`evaluate()`, in `02_ga.ipynb`) built independently
by Dhammika from the rules alone.